# FW-LNSA final research validation

Run in a fresh **CPU** Google Colab session with Python 3.12. This notebook installs an isolated scientific environment and uses the new CLI; it never modifies the manuscript or primary evidence. Original seeds100–119 are replayed only with saved locks. Every sensitivity is secondary. Runtime depends on CPU/RAM: allow several hours for all real-data jobs, especially30 stability fits per dataset and120k tuning. Use persistent Drive output for disconnect recovery.

Upload this notebook plus the delivered code ZIP, original starter ZIP, and original data ZIP to `/content`, or set their paths below. Keep at least8GB free RAM; a high-RAM CPU runtime is helpful. GPU acceleration is not used. Do not publish raw datasets or private `.pkl` checkpoints.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, zipfile
LOCAL_TEST = os.environ.get("FW_VALIDATION_LOCAL_TEST") == "1"
if sys.version_info[:2] != (3, 12):
    raise RuntimeError("This locked workflow requires Python 3.12. Select a compatible Colab runtime or use the Python 3.12 CLI; do not silently change scientific versions.")
PROFILE = "smoke" if LOCAL_TEST else "full"  # explicit; smoke never proves real-data jobs
INCLUDE_120K = True
MOUNT_DRIVE = not LOCAL_TEST
BASE_COMMIT = "8496b94717c85566748c767c387f999d76bdc9a7"
REPO_ROOT = Path(os.environ.get("FW_VALIDATION_TEST_REPO", "/content/FW-LNSA-NIDS-final-validation"))
INPUT_ROOT = Path("/content")
CODE_ZIP = INPUT_ROOT / "FW_LNSA_Final_Research_Code_GitHub_Ready.zip"
STARTER_ZIP = None  # None discovers one FW_LNSA_Work_Starter_Bundle*.zip in INPUT_ROOT
RAW_DATA_ZIP = None  # None discovers one FW-LNSA Paper Writing*.zip in INPUT_ROOT
OUTPUT_ROOT = Path(os.environ.get("FW_VALIDATION_TEST_OUTPUT", "/content/drive/MyDrive/FW-LNSA-final-validation/full_v1"))
if LOCAL_TEST:
    ACCEPTED_ROOT = Path(os.environ["FW_VALIDATION_TEST_ACCEPTED"])
    DATA_ROOT = Path(os.environ["FW_VALIDATION_TEST_DATA"])
    RUNTIME_PYTHON = Path(sys.executable)
print({"profile": PROFILE, "include_120k": INCLUDE_120K, "output": str(OUTPUT_ROOT)})

## Prepare code and persistent outputs

The setup refuses an incorrect base or a dirty clone. If you already applied the release, reruns verify the base and reuse it. A changed source/profile/environment requires a new versioned output folder. Preserve the old folder for reconciliation.

In [ ]:
def run(command, cwd=None, env=None):
    subprocess.run([str(x) for x in command], cwd=cwd, env=env, check=True)

def safe_extract(archive_path, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError("Unsafe ZIP member")
        archive.extractall(destination)

def discover(path, pattern):
    if path is not None:
        return Path(path)
    candidates = sorted(INPUT_ROOT.glob(pattern))
    if len(candidates) != 1:
        raise ValueError(f"Set an explicit path; found {len(candidates)} matches for {pattern}")
    return candidates[0]

if not LOCAL_TEST:
    if MOUNT_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
    if not CODE_ZIP.is_file():
        raise FileNotFoundError(f"Upload delivered code package: {CODE_ZIP}")
    package = INPUT_ROOT / "fw_lnsa_release_package"
    safe_extract(CODE_ZIP, package)
    if not REPO_ROOT.exists():
        run(["git", "clone", "--branch", "main", "https://github.com/sarosh-jawed/FW-LNSA-NIDS.git", REPO_ROOT])
        run(["git", "checkout", "--detach", BASE_COMMIT], cwd=REPO_ROOT)
        run([sys.executable, package / "APPLY_TO_CLONE.py", "--repo", REPO_ROOT])
    else:
        actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, text=True).strip()
        if actual != BASE_COMMIT:
            raise ValueError("Existing clone is not at pinned base; use a new clean folder")
        # Verify every delivered overlay byte before reuse; no copying over edits.
        manifest = json.loads((package / "CHANGED_FILES.json").read_text())
        import hashlib
        for row in manifest:
            target = REPO_ROOT / row["path"]
            if not target.is_file() or hashlib.sha256(target.read_bytes()).hexdigest() != row["sha256"]:
                raise ValueError(f"Existing applied source differs: {target}")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Prepared code and output folder")

## Install the locked environment and locate immutable inputs

The scientific code runs in its own virtual environment, avoiding preloaded Colab packages. The historical scientific library versions remain pinned. Only trusted attached archives are unpacked locally. Their hashes are independently checked before empirical jobs.

In [ ]:
if not LOCAL_TEST:
    runtime = REPO_ROOT / ".venv-final-validation"
    if not runtime.exists():
        run([sys.executable, "-m", "venv", runtime])
    RUNTIME_PYTHON = runtime / "bin/python"
    run([RUNTIME_PYTHON, "-m", "pip", "install", "-r", "requirements-final-validation-lock.txt"], cwd=REPO_ROOT)
    starter_path = discover(STARTER_ZIP, "FW_LNSA_Work_Starter_Bundle*.zip")
    raw_path = discover(RAW_DATA_ZIP, "FW-LNSA Paper Writing*.zip")
    private = INPUT_ROOT / "fw_lnsa_private_inputs"
    safe_extract(starter_path, private / "starter")
    safe_extract(private / "starter/accepted_evidence/writer_handoff.zip", private / "accepted")
    safe_extract(raw_path, private / "raw")
    ACCEPTED_ROOT = private / "accepted"
    DATA_ROOT = private / "raw/FW-LNSA Paper Writing/datasets_reference"
THREAD_ENV = dict(os.environ)
THREAD_ENV.update({k: "1" for k in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"]})
run([RUNTIME_PYTHON, "-m", "pip", "check"], cwd=REPO_ROOT, env=THREAD_ENV)
print("Inputs:", ACCEPTED_ROOT, DATA_ROOT)

## Software tests and evidence preflight

Tests create synthetic fixtures. Passing them proves execution paths, not the remaining experiments. The evidence job verifies immutable accepted files,480 FW-LNSA/960 reference seed rows, original120-test statistics and calibration traceability.

In [ ]:
run([RUNTIME_PYTHON, "-m", "pytest", "-q"], cwd=REPO_ROOT, env=THREAD_ENV)
run([RUNTIME_PYTHON, "-m", "ruff", "check", "src/final_validation", "scripts/run_final_validation.py", "scripts/verify_final_validation.py", "tests/test_final_validation.py"], cwd=REPO_ROOT, env=THREAD_ENV)
COMMON = [RUNTIME_PYTHON, "scripts/run_final_validation.py", "--profile", PROFILE,
          "--accepted", ACCEPTED_ROOT, "--data-root", DATA_ROOT, "--output", OUTPUT_ROOT]
run(COMMON + ["--jobs", "evidence"], cwd=REPO_ROOT, env=THREAD_ENV)

## Run new analyses in dependency order

The main data jobs must reproduce every archived partition and selected-feature hash. Thirty training resamples, no-port/mRMR/self-cap/k sensitivities, frozen pool/category/score-tie replay and controlled benchmarks follow. Per-resample, per-seed and per-benchmark-repeat files resume after disconnection. Rerun this cell with the same exact code/environment/inputs and output folder. Never change the original locks to improve test scores.

The120k study has new tuning seeds200–204 and evaluation seeds300–319, a deterministic validation-only tie rule and separately reserved test partition. It may overlap60k; it is not external validation. When disabled its status remains pending.

In [ ]:
jobs = ["data", "stability", "development", "replay", "benchmark"]
if INCLUDE_120K:
    jobs.append("sample120k")
jobs.append("report")
run(COMMON + ["--jobs"] + jobs + (["--include-120k"] if INCLUDE_120K else []), cwd=REPO_ROOT, env=THREAD_ENV)

## Independently verify and export

An export includes hashes and public evidence but omits raw feature arrays, fitted pickles and caches. A normal result export may remain PRELIMINARY even when software passes. Retain the private output folder if you need to resume. Bring the export and verification JSON back for independent review before GitHub push or final coauthor handoff.

In [ ]:
EXPORT_ZIP = OUTPUT_ROOT.parent / (OUTPUT_ROOT.name + "_results_export.zip")
run([RUNTIME_PYTHON, "scripts/make_final_secondary_report.py", "--root", OUTPUT_ROOT, "--profile", PROFILE], cwd=REPO_ROOT, env=THREAD_ENV)
run([RUNTIME_PYTHON, "scripts/validate_final_outputs.py", "--root", OUTPUT_ROOT, "--accepted", ACCEPTED_ROOT, "--profile", PROFILE], cwd=REPO_ROOT, env=THREAD_ENV)
run(COMMON + ["--jobs", "report", "--export", EXPORT_ZIP], cwd=REPO_ROOT, env=THREAD_ENV)
run([RUNTIME_PYTHON, "scripts/verify_final_validation.py", "--export", EXPORT_ZIP,
     "--output", OUTPUT_ROOT.parent / (OUTPUT_ROOT.name + "_independent_checks.json")], cwd=REPO_ROOT, env=THREAD_ENV)
import pandas as pd
from IPython.display import display
display(pd.read_csv(OUTPUT_ROOT / "status.csv"))
print("Export ready:", EXPORT_ZIP)
if not LOCAL_TEST:
    from google.colab import files
    files.download(str(EXPORT_ZIP))